In [ ]:
# TODAY, WE INTRODUCE LANG-GRAPH (Formalising my agent as a state machine)

"""Today is the conceptual pivot of the entire month. Everything I built manually
across days 8-13, the ReAct loop, the CRAG pipeline, the router, the reviewer all gets expressed
as a graph. Nodes are actions. Edges are transitions. State is a types dictionary 
that flows through the graph and accumulates results at every node. By the end of day
today, I will have rebuilt my router pipeline as a proper LangGraph graph, and the 
manual while loops from days 8-10 will never be necessary again.

The State Schema: Define a typed typeDict that represents everything my graph needs
to track the Question, retrieved context, generated answer, reviewer verdict, retrieval source,
rewrite count, and classification. This replaces the ad-hoc result dictionaries from
days 11-12

The Nodes: Convert each major function from days 11-12 into a LangGraph node-A plain 
python function that receives the current state and returns a partial state update.
You will build five nodes: classify_node, retrieve_node, generate_node, review_node, and rewrite_node

The Edges: wire the nodes together. Most edges are fixed (classify->retrieve->generate->review).
The Edge out of review_node is conditional, if the verdict is PASS it goes to END, if FAIL and rewrites
remain, it loops back to rewrite_node, if max rewrites are exhausted it also goes to END with a warning flag.

Compile and Run: Call graph.compile() and invoke it with a question. Watch the 
state flow through every node and arrive at END fully populated.

"""

# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"
    

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ],

        temperature= 0,
    )

    return response.choices[0].message.content


def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-120b',
        messages = [
            {"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
            {"role": "user", "content":(
                f"Question: {question}\n\n"
                f"Source Context:\n{context}\n\n"
                f"Generated Answer:\n{answer}"

            )}
        ],
        temperature= 0
    )

    raw = response.choices[0].message.content
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason

# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }


# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result



Unable to verify collection 'month_1_rag_collection_v3' access: SSL handshake failed: ac-2ma9ajs-shard-00-01.clmmfwh.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1016) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-2ma9ajs-shard-00-02.clmmfwh.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1016) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-2ma9ajs-shard-00-00.clmmfwh.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1016) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6a42b900e4def33c01c2d8bc, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-2ma9ajs-shard-00-00.clmmfwh.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnec

🤖🛩️ Vector Store connection established ⚡


In [13]:
# THE STATE SCHEMA

class RAGState(TypedDict):
    # Input
    question: str

    # Router output
    classification: Optional[str]
    classification_reason: Optional[str]

    # Retrieval output
    context: Optional[str]
    retrieval_source: Optional[str]
    retrieval_score: Optional[float]

    # Generation output
    answer: Optional[str]

    # Reviewer output
    reviewer_verdict: Optional[str]
    reviewer_reason: Optional[str]

    # Rewrite tracking
    rewrite_count: int
    max_rewrites: int

    # Final flags
    warning: Optional[str]
    finished_at: Optional[str]


In [14]:
# THE FIVE NODES

# -----Node 1: Classify -------
def classify_node(state: RAGState) -> dict:
    print(f"\n[NODE: classify] Question: {state['question'][:60]}...")
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Question: {state['question']}"}],
        system= ROUTER_SYSTEM_PROMPT
    )

    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not parse."

    print(f" -> Classification: {classification} - {reason}")
    return {"classification": classification, "classification_reason": reason}

# ------ Node 2: Retrieve ------
def retrieve_node(state: RAGState) -> dict:
    print(f"\n[NODE: retrieve]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN query - skipping retrieval")
        return{
            "context": "No context retrieved - query classified as UNKNOWN.",
            "retrieval_source": "none",
            "retrieval_score": 0.0

        }
    
    docs, score = retrieve_with_confidence(state["question"])

    if score < RELEVANCE_THRESHOLD or not docs:
        print(" -> CRAG: Low confidence - falling back to web search")
        web_results = webSearch.invoke(state["question"])
        context = "\n\n".join([r["content"] for r in web_results])
        return {"context": context, "retrieval_source": "web", "retrieval_score": score}
    
    context = format_chunks(docs= docs)
    print(f" -> Local retrieval accepted (score: {score:.3f})")
    return {"context": context, "retrieval_source": "local", "retrieval_score": score}


# ---Node 3: Generate ----
def generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: generate]")

    if state["classification"] == "UNKNOWN":
        answer = (
            "This question cannot be answered using the Apple FY2024 10-K document"
            "or available tools."
        )
        print(f" -> UNKNOWN path - returning abstention")
        return {"answer": answer}
    
    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    answer = tracked_llm_call(
        messages = [{
            "role":"user",
            "content": (
                f"Context:\n{state['context']}\n\n"
                f"Question: {state['question']}"
                f"{critique_block}"
            )

        }],
        system= GENERATOR_SYSTEM_PROMPT
    )
    print(f" -> Answer generated ({len(answer)} chars)")
    return {"answer": answer}


# -----Node 4: Review ----
def review_node(state: RAGState) -> dict:
    print(f"\n[NODE: review]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN path - skipping review")
        return {"reviewer_verdict": "PASS", "reviewer_reason": "Abstention accepted."}
    
    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Source Context:\n{state['context']}\n\n"
                f"Generated Answer:\n{state['answer']}"
            )
        }],
        system = REVIEWER_SYSTEM_PROMPT
    )

    verdict_match =re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Reviewer: {verdict} - {reason}")
    return {"reviewer_verdict" : verdict, "reviewer_reason": reason}


# ---- Node 5: Rewrite ----
def rewrite_node(state: RAGState) -> dict:
    new_count = state.get("rewrite_count", 0) + 1
    print(f"\n[NODE: rewrite] Attempt {new_count}")
    return {
        "rewrite_count" : new_count,
        "answer": None # cleared so generate_node produces a fresh answer
    }


    



In [15]:
# CONDITIONAL EDGE LOGIC

def route_after_review(state: RAGState) -> Literal["rewrite_node", "__end__"]:
    """
    Called after review_node .
    Decides whether to loop back for a rewrite or proceed to END.

    """

    verdict = state.get("reviewer_verdict", "FAIL")
    rewrite_count = state.get("rewrite_count", 0)
    max_rewrites = state.get("max_rewrites", 2)

    if verdict == "PASS":
        print(" -> Edge: PASS -> END")
        return "__end__"
    
    if rewrite_count >= max_rewrites:
        print(f" -> Edge: max rewrites ({max_rewrites}) reached -> END with warning")
        return "__end__"
    
    print(f" -> Edge: FAIL -> rewrite_node (attempt {rewrite_count + 1})")
    return "rewrite_node"

def route_after_classify(state: RAGState) -> Literal["retrieve_node"]:
    """
    All paths go through retrieve_node - it handles UNKNOWN internally.
    This edge exists to make the graph structure explicit.
    """
    return "retrieve_node"

In [ ]:
# BUILDING AND COMPILING THE GRAPH

def build_rag_graph() -> StateGraph:
    graph = StateGraph(RAGState)

    # Adding all nodes
    graph.add_node("classify_node", classify_node)
    graph.add_node("retrieve_node", retrieve_node)
    graph.add_node("generate_node", generate_node)
    graph.add_node("review_node", review_node)
    graph.add_node("rewrite_node", rewrite_node)

    # Set entry point
    graph.set_entry_point("classify_node")

    # Fixed edges
    graph.add_edge("classify_node", "retrieve_node")
    graph.add_edge("retrieve_node", "generate_node")
    graph.add_edge("generate_node", "review_node")
    graph.add_edge("rewrite_node", "generate_node")

    # Conditional edge out of review_node
    graph.add_conditional_edges(
        "review_node",
        route_after_review,
        {
            "rewrite_node": "rewrite_node",
            "__end__": END
        }
    )

    return graph.compile()

rag_graph = build_rag_graph()

# visualise the graph structure
try:
    print(rag_graph.get_graph().draw_ascii())
except Exception:
    print("ASCII visualisation unavailable - install pygraphviz for full diagram")

ASCII visualisation unavailable - install pygraphviz for full diagram


In [17]:
# THE GRAPH RUNNER
def run_graph(question: str, max_rewrites: int = 2) -> RAGState:
    global usage_tracker
    usage_tracker = TokenUsage()

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    initial_state: RAGState = {
        "question": question,
        "classification": None,
        "classification_reason": None,
        "context": None,
        "retrieval_source": None,
        "retrieval_score": None,
        "answer": None,
        "reviewer_verdict": None,
        "reviewer_reason": None,
        "rewrite_count": 0,
        "max_rewrites": max_rewrites,
        "warning": None,
        "finished_at": None,
    }

    final_state = rag_graph.invoke(initial_state)
    final_state["finished_at"] = datetime.now().isoformat()

    usage_tracker.report()

    print(f"\n{'='*60}")
    print(f"😁 Final Answer:")
    print(final_state["answer"])
    print(f"{'='*60}\n")

    os.makedirs("traces", exist_ok= True)


    filename = f"traces/day14_graph_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(dict(final_state), f, indent = 2)
    print(f"📀 Saved to {filename}")

    return final_state



In [19]:
 # EVALUATION TIME

# Same three questions as day11 - outputs must match

# Question 1: High confidence local retrieval
run_graph(
    "What was Apple's total net sales for fiscal year 2024 "
    "and how does it compare to fiscal year 2023?"
)

# Question 2: Borderline retrieval - reviewer may trigger rewrite
run_graph(
    "What were the primary risk factors Apple disclosed in the FY2024 10-K"
    "related to global macroeconomic conditions?"
)

# Question 3: Outside the 10-K - CRAG web fallback must fire
run_graph(
    "What is Apple's current stock price and market capitalisation today?"
)


Question: What was Apple's total net sales for fiscal year 2024 and how does it compare to fiscal year 2023?

[NODE: classify] Question: What was Apple's total net sales for fiscal year 2024 and ho...
 -> Classification: COMPLEX - The query needs two separate data lookups (FY2024 and FY2023 net sales) and a comparison between them.

[NODE: retrieve]
📣 Top retrieval score : 0.883 (threshold: 0.5)
 -> Local retrieval accepted (score: 0.883)

[NODE: generate]
 -> Answer generated (251 chars)

[NODE: review]
 -> Reviewer: PASS - The answer correctly reports the total net sales for FY 2024 and FY 2023 and accurately calculates the difference and percentage increase, all supported by the provided context.
 -> Edge: PASS -> END

📣 Token usage report
LLM calls : 3
Prompt tokens: 2767
Completion tokens: 402
Total tokens: 3,169
Est. cost (GPT-40 pricing): $0.010937

😁 Final Answer:
Apple’s total net sales for fiscal year 2024 were **$391,035 million**.  
For fiscal year 2023, total net sales we

{'question': "What is Apple's current stock price and market capitalisation today?",
 'classification': 'UNKNOWN',
 'classification_reason': 'The request requires up-to-the-minute market data that is not available in static financial documents.',
 'context': 'No context retrieved - query classified as UNKNOWN.',
 'retrieval_source': 'none',
 'retrieval_score': 0.0,
 'answer': 'This question cannot be answered using the Apple FY2024 10-K documentor available tools.',
 'reviewer_verdict': 'PASS',
 'reviewer_reason': 'Abstention accepted.',
 'rewrite_count': 0,
 'max_rewrites': 2,
 'warning': None,
 'finished_at': '2026-06-29T21:33:05.186982'}